# Score New Resumes

This notebook demonstrates how to train the `ResumeMLPipeline` and compute scores for new resumes.

In [ ]:
import numpy as np
import pandas as pd

from ml_pipeline import ResumeMLPipeline


In [ ]:
# Train pipeline on existing datasets (same as score_new_resumes.py)
resumes_path = 'UpdatedResumeDataSet_cleaned.csv'
structured_path = 'resume_shortlisting_dataset_v2.csv'

resumes_df = pd.read_csv(resumes_path)
structured_df = pd.read_csv(structured_path)

text_col = 'Resume'
label_col = 'Category'

texts = resumes_df[text_col].astype(str).tolist()
labels = resumes_df[label_col]

pipeline = ResumeMLPipeline()
semantic_vecs = pipeline.encode_texts(texts)
structured_vecs = pipeline.build_structured_features(structured_df)
final_vectors = pipeline.fuse_features(semantic_vecs, structured_vecs)
pipeline.fit_clusters(final_vectors, labels=labels)

print('Training complete.')


In [ ]:
# Helper to score a single resume in this notebook
def score_resume(resume_text: str, years_experience: float, projects_count: float, certificates_count: float):
    semantic_vec = pipeline.encode_texts([resume_text])[0]
    structured_vec = pipeline.transform_structured_row(
        years_experience=years_experience,
        projects_count=projects_count,
        certificates_count=certificates_count,
    )
    fused_vec = pipeline.fuse_features(semantic_vec.reshape(1, -1), structured_vec)[0]
    sims, sims_norm = pipeline.skill_similarity(fused_vec)
    best_idx = int(np.argmax(sims_norm))
    S = float(sims_norm[best_idx])
    E_norm, P_norm, C_norm = structured_vec[0]
    score = pipeline.compute_scores(S=S, E=E_norm, P=P_norm, C=C_norm)
    is_hybrid, top1, top2, confidence = pipeline.detect_hybrid(sims)
    return {
        'score': score,
        'best_profession_index': best_idx,
        'normalized_similarity': sims_norm.tolist(),
        'E_norm': float(E_norm),
        'P_norm': float(P_norm),
        'C_norm': float(C_norm),
        'is_hybrid': bool(is_hybrid),
        'top1_index': int(top1),
        'top2_index': int(top2),
        'confidence_gap': float(confidence),
    }


In [ ]:
# Example: fill in your own resume text and structured info here
example_resume = 'Experienced data scientist with 5 years in machine learning and NLP.'
years_experience = 5.0
projects_count = 3.0
certificates_count = 2.0

result = score_resume(example_resume, years_experience, projects_count, certificates_count)
result
